In [2]:
import os

import s3fs
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient, models

load_dotenv(override=True)

# Themes
THEME_FILTER = ["Heures supplémentaires (contingent, majoration)"]

# Data storage
fs = s3fs.S3FileSystem(
    endpoint_url="https://minio.lab.sspcloud.fr",
    client_kwargs={"region_name": "us-east-1"},
)
FULL_DATA_PATH = "s3://mateomorin/legifrance/data/"

# Embedding creation
EMBEDDING_MODEL = "qwen3-embedding-8b"
EMBEDDING_SIZE = 4096               # to be changed depending on the model

llm_client = OpenAI(
    base_url=os.environ["LLM_API_URL"],
    api_key=os.environ["LLM_API_KEY"],
)


# Vector storage
qdrant_client = QdrantClient(
    url=os.environ.get("QDRANT_API_URL", "http://qdrant:6333"),
    api_key=os.environ.get("QDRANT_API_KEY", None),
    timeout=60
)

/home/onyxia/work/accords_heures_supp/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:758: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is empty, alternative syntax for `is_empty: \&quot;field_name\&quot;`",
/home/onyxia/work/accords_heures_supp/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:762: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is null, alternative syntax for `is_null: \&quot;field_name\&quot;`",
/tmp/ipykernel_105543/1437346810.py:31: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


In [3]:
if not qdrant_client.collection_exists(collection_name="test"):

    qdrant_client.create_collection(
        collection_name="test",
        vectors_config=models.VectorParams(
            size=4096,
            distance=models.Distance.COSINE
        )
    )

else:
    print("Collection already exists")

Collection already exists


In [4]:
EMBEDDING_MODEL = "qwen3-embedding-8b"

chunks = [
    "Je suis une autruche et je mange des graines",
    "Je suis un lion et je rugis",
    "Je suis un chat et je chasse une souris"
]

embeddings = llm_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=chunks
)

In [7]:
embeddings.data[1]

Embedding(embedding=[0.017336705699563026, -0.00237347767688334, 0.015685591846704483, -0.026889584958553314, 0.01090915221720934, -0.03089943528175354, -0.004923860542476177, 0.008314542472362518, -0.0033906823955476284, 0.0024914145469665527, -0.021700367331504822, -0.025356408208608627, 0.04316486045718193, -0.011793677695095539, 0.021228620782494545, 0.016039401292800903, 0.003921397961676121, 0.012855108827352524, 0.016747022047638893, 0.053543299436569214, -0.022054176777601242, 0.02346941828727722, 0.016747022047638893, -0.00403933459892869, -0.009611847810447216, 0.05401504412293434, -0.00518921809270978, 0.004864891991019249, -0.027125459164381027, -0.00648652296513319, 0.027833079919219017, -0.023705292493104935, -0.019105758517980576, 0.021582430228590965, 0.004068818874657154, 0.008904227055609226, -0.03514515981078148, -0.007430016994476318, 0.009375973604619503, -0.009965658187866211, -0.020992746576666832, -0.015921466052532196, 0.02523847110569477, 0.014093444682657719,

In [12]:
import uuid

qdrant_client.upsert(
    collection_name="test",
    points=[
        models.PointStruct(
            id=str(uuid.uuid1()),
            payload={
                "color": "red",
            },
            vector=embeddings.data[0].embedding,
        ),
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
qdrant_client.upsert(
    collection_name="test",
    points=models.Batch(
        ids=[1, 2, 3],
        payloads=[
            {"color": "red"},
            {"color": "green"},
            {"color": "blue"},
        ],
        vectors=[
            embeddings.data[0].embedding,
            embeddings.data[1].embedding,
            embeddings.data[2].embedding,
        ],
    )
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [14]:
import pandas as pd

df = pd.read_parquet("s3://mateomorin/legifrance/data/acco_data_2025/shard_00000.parquet", filesystem=fs)

In [19]:
metadata = df.drop(columns=["markdown"]).to_dict(orient="index").values()

In [23]:
qdrant_client.create_collection(
    collection_name="heures_supp_2017",
    vectors_config=models.VectorParams(size=4096, distance=models.Distance.COSINE),
)

True

In [24]:
SOURCE_COLLECTION = "acco_data_2017_9_10_11_12"
TARGET_COLLECTION = "heures_supp_2017"
BATCH_SIZE = 500  # Nombre de points traités par lot

# 1. Récupération de la configuration de la collection source
source_info = qdrant_client.get_collection(collection_name=SOURCE_COLLECTION)
vectors_config = source_info.config.params.vectors

# 2. Création de la collection cible avec la même configuration
if not qdrant_client.collection_exists(TARGET_COLLECTION):
    qdrant_client.create_collection(
        collection_name=TARGET_COLLECTION,
        vectors_config=vectors_config
    )
    print(f"Collection '{TARGET_COLLECTION}' créée.")

# 3. Copie des points par lots
next_page_offset = None
total_copied = 0

while True:
    # Récupération d'un lot de points (vecteurs + payloads)
    records, next_page_offset = qdrant_client.scroll(
        collection_name=SOURCE_COLLECTION,
        limit=BATCH_SIZE,
        offset=next_page_offset,
        with_payload=True,
        with_vectors=True
    )

    if not records:
        break

    # Préparation des points à réinsérer
    points_to_upsert = [
        models.PointStruct(
            id=record.id,
            vector=record.vector,
            payload=record.payload
        )
        for record in records
    ]

    # Insertion dans la nouvelle collection
    qdrant_client.upsert(
        collection_name=TARGET_COLLECTION,
        points=points_to_upsert
    )

    total_copied += len(records)
    print(f"Points copiés : {total_copied}")

    # Arrêt lorsqu'il n'y a plus de pages
    if next_page_offset is None:
        break

print(f"Migration terminée. {total_copied} points ont été transférés vers '{TARGET_COLLECTION}'.")

Points copiés : 500
Points copiés : 1000
Points copiés : 1500
Points copiés : 2000
Points copiés : 2500
Points copiés : 3000
Points copiés : 3500
Points copiés : 4000
Points copiés : 4500
Points copiés : 5000
Points copiés : 5500
Points copiés : 6000
Points copiés : 6500
Points copiés : 7000
Points copiés : 7500
Points copiés : 8000
Points copiés : 8500
Points copiés : 9000
Points copiés : 9500
Points copiés : 10000
Points copiés : 10500
Points copiés : 11000
Points copiés : 11500
Points copiés : 12000
Points copiés : 12500
Points copiés : 13000
Points copiés : 13500
Points copiés : 14000
Points copiés : 14500
Points copiés : 15000
Points copiés : 15500
Points copiés : 16000
Points copiés : 16500
Points copiés : 17000
Points copiés : 17500
Points copiés : 18000
Points copiés : 18061
Migration terminée. 18061 points ont été transférés vers 'heures_supp_2017'.


In [30]:
pd.to_datetime(df["dateDiffusion"]).dt.strftime("%Y-%m-%d")

0       2025-01-20
1       2025-01-20
2       2025-02-03
3       2025-02-03
4       2025-02-03
           ...    
1995    2025-02-24
1996    2025-02-24
1997    2025-02-24
1998    2025-02-24
1999    2025-02-24
Name: dateDiffusion, Length: 2000, dtype: str

In [22]:
list(metadata)

[{'cid': 'ACCOTEXT000051004200',
  'id': 'ACCOTEXT000051004200',
  'title': "Accord en faveur de l'Egalité Professionnelle et de la promotion de la mixité",
  'legalStatus': None,
  'startDate': None,
  'endDate': None,
  'nature': None,
  'reference': 'T00625062185',
  'dateSignature': Timestamp('2025-01-01 00:00:00+0000', tz='UTC'),
  'dateDiffusion': '2025-01-20T00:00:00.000+0000',
  'conforme': True,
  "Accord de maintien dans l'emploi (à venir)": False,
  'Accords de méthode (PSE)': False,
  'Accords de méthode (pénibilité)': False,
  'Aménagement du temps de travail (modulation, annualisation, cycles)': False,
  'Autre, précisez': False,
  'Autres dispositions Egalité professionnelle': True,
  'Autres dispositions de conditions de travail (CHSCT, médecine du travail, politique générale de prévention)': False,
  'Autres dispositions durée et aménagement du temps de travail': False,
  'Autres dispositions durée et aménagement du temps de travail ': False,
  'Autres dispositions emp